# 12.3: Memberi Tangan pada LLM

Model bahasa hanya bisa menghasilkan teks. Ia tidak bisa menghitung dengan
tepat, dan tidak tahu apa pun yang terjadi setelah data latihnya dibekukan.

Tool calling menutup kedua lubang itu. Anda memberi model sebuah daftar fungsi
Python, model memutuskan mana yang perlu dipanggil, dan **kode Anda** yang
menjalankannya.

Bedanya dengan Sesi 11 hanya satu argumen tambahan. Sungguh.

In [1]:
import json
import os
import sys
import urllib.request

sys.path.insert(0, "..")

import ollama
client = ollama.Client()

MODEL = "gpt-oss:20b"
print("siap")

siap


## 1. Kita coba ajukan dua pertanyaan berbeda

Kita akan menyuruh LLM untuk menghitung dan memberikan prakiraan cuaca

In [3]:
soal_list = [
    "Berapa hasil 1247 dikali 89?",
    "Berapa suhu di Jakarta sekarang?"
    ]

for soal in ["Berapa hasil 1247 dikali 89?",
             "Berapa suhu di Jakarta sekarang?"]:
    resp = client.chat(
        model=MODEL, 
        messages=[
            {
                "role": "user",
                "content": soal
            }
        ],
        think=False
    )
    print(f"T: {soal}")
    print(f"J: {resp.message.content.strip()}")
    print()

print("hitungan sebenarnya:", 1247 * 89)

T: Berapa hasil 1247 dikali 89?
J: 1247 × 89 = 110 983

T: Berapa suhu di Jakarta sekarang?
J: Maaf, saya tidak dapat mengakses data cuaca secara real‑time. Untuk mengetahui suhu Jakarta saat ini, Anda bisa memeriksa aplikasi cuaca di smartphone (misalnya Weather, AccuWeather, atau aplikasi cuaca resmi BMKG) atau situs web berita lokal yang biasanya menampilkan perkiraan cuaca terkini.

Secara umum, di Jakarta suhu biasanya berkisar antara 27 °C hingga 32 °C, tergantung musim dan kondisi cuaca hari itu. Semoga membantu!

hitungan sebenarnya: 110983


## 2. Tool pertama, sebuah kalkulator

Sebuah tool adalah fungsi Python biasa. Tidak ada kelas khusus, tidak ada
dekorator, tidak ada pendaftaran.

Dua hal yang dibaca model dari fungsi Anda:

- **docstring** menjadi deskripsi, yang dipakai model untuk memutuskan kapan
  fungsi ini relevan,
- **type hint** menjadi schema, yang dipakai model untuk menyusun argumennya.

Keduanya bukan hiasan. Docstring yang malas menghasilkan tool yang jarang
dipanggil, atau dipanggil di saat yang salah.

Satu catatan praktis: jaga agar parameternya tetap sederhana, cukup `str`,
`int`, atau `float`. Parameter berbentuk list atau dict sering sampai dalam
keadaan tidak rapi dan menyulitkan pemula.

In [4]:
def kalkulator(a: float, b: float, operator: str) -> str:
    # TODO 1: tulis docstring yang menjelaskan fungsi ini dan ketiga
    # argumennya. Docstring inilah yang dibaca model untuk memutuskan
    # kapan tool ini relevan, jadi tulis dengan jelas.
    if operator == "/" and b == 0:
        return "tidak terdefinisi, pembagi nol"
    hasil = {"+": a + b, "-": a - b, "*": a * b,
             "/": a / b if b else 0.0}[operator]
    return str(hasil)


print(kalkulator(1247, 89, "*"))

110983


In [7]:
def kalkulator(a: float, b: float, operator: str) -> str:
    """ 
    Hitung operasi aritmatika dari dua angka

    Args:
        a: angka pertama
        b: angka kedua
        operator: salah satu dari "+", "-", "/", "*"
    """
    if operator == "/" and b == 0:
        return "tidak terdefinisi, pembagi nol"
    hasil = {"+": a + b, "-": a - b, "*": a * b,
             "/": a / b if b else 0.0}[operator]
    return str(hasil)


print(kalkulator(1247, 89, "*"))

110983


Sekarang serahkan fungsi itu kepada model. Inilah satu argumen yang menjadi
seluruh pelajaran hari ini:

```python
config=types.GenerateContentConfig(tools=[kalkulator])
```

SDK akan mengubah fungsi Anda menjadi schema, menjalankan fungsinya ketika
model memintanya, lalu mengirimkan hasilnya kembali ke model. Semua itu terjadi
di dalam satu pemanggilan.

In [5]:
import ollama

def kalkulator(ekspresi: str) -> str:
    """Menghitung ekspresi matematika sederhana."""
    return str(eval(ekspresi))

tools = [kalkulator]

resp = client.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Berapa hasil 1247 dikali 89?"
        }
    ],
    tools=tools
)

print(resp.message.content.strip())

Hasil perkalian 1247 × 89 adalah **110 983**.


## 3. Tool kedua, data dari luar

Kalkulator menyelesaikan masalah hitungan. Masalah kedua adalah model tidak tahu
apa pun tentang hari ini.

Kita pakai Open-Meteo, yang tidak memerlukan API key maupun pendaftaran. Ia
meminta koordinat, bukan nama kota, jadi tool kita membawa kamus kecil. Ini
sekaligus menunjukkan bahwa sebuah tool boleh melakukan apa saja yang bisa
dilakukan Python, termasuk mencari di tabel.

Perhatikan `try` dan `except`. Sebuah tool yang gagal sebaiknya mengembalikan
kalimat, bukan melempar error. Kalau ia melempar, seluruh alur berhenti. Kalau
ia mengembalikan kalimat, model bisa menjelaskan keadaannya kepada pengguna.

In [8]:
KOTA = {
    "Jakarta": (-6.21, 106.85),
    "Bandung": (-6.91, 107.61),
    "Surabaya": (-7.25, 112.75),
    "Medan": (3.59, 98.67),
    "Makassar": (-5.15, 119.43),
}


def cuaca(kota: str) -> str:
    """Ambil suhu udara terkini sebuah kota di Indonesia.

    Args:
        kota: nama kota, misalnya "Jakarta" atau "Bandung"
    """
    if kota not in KOTA:
        return f"Kota {kota} tidak ada dalam daftar: {', '.join(KOTA)}"
    lat, lon = KOTA[kota]
    url = (f"https://api.open-meteo.com/v1/forecast?latitude={lat}"
           f"&longitude={lon}&current=temperature_2m")
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            data = json.loads(r.read())
        suhu = data["current"]["temperature_2m"]
        return f"Suhu di {kota} saat ini {suhu} derajat Celcius."
    except Exception as e:
        return f"Gagal mengambil data cuaca: {e}"


print(cuaca("Jakarta"))

Suhu di Jakarta saat ini 27.1 derajat Celcius.


## 4. Dua tool, model yang memilih sendiri

Satu tool mengajarkan mekanismenya. Dua tool mengajarkan konsep sebenarnya:
model **memilih**.

Jalankan tiga pertanyaan berikut. Tidak satu pun dari kita yang memberi tahu
model tool mana yang harus dipakai.

In [9]:
# TODO 3: susun satu GenerateContentConfig yang membawa KEDUA tool,
# lalu jalankan ketiga pertanyaan di bawah ini dengan config itu.

TOOLS = [kalkulator, cuaca]

for soal in [
    "Berapa 15 persen dari 50 juta?",
    "Bagaimana cuaca di Bandung hari ini?",
    "Halo, kamu bisa bantu apa saja?"
]:
    resp = client.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": soal
            }
        ],
        tools=TOOLS
    )

    print(f"T: {soal}")

    if resp.message.tool_calls:
        print("Tool yang diminta model:")
        for tool_call in resp.message.tool_calls:
            print(f"- {tool_call.function.name}")
    else:
        print(f"J: {resp.message.content.strip()[:220]}")

    print()

T: Berapa 15 persen dari 50 juta?
J: 15 % dari 50 juta = 0,15 × 50 000 000 = 7 500 000.  
Jadi hasilnya **7,5 juta**.

T: Bagaimana cuaca di Bandung hari ini?
Tool yang diminta model:
- cuaca

T: Halo, kamu bisa bantu apa saja?
J: Halo! Saya bisa membantu dengan berbagai hal, mulai dari:

- Menjawab pertanyaan dan memberikan penjelasan
- Membantu menulis, mengedit, atau menerjemahkan teks
- Menyusun rencana, daftar tugas, atau outline tulisan
- Me



Pertanyaan ketiga tidak memanggil tool sama sekali, karena memang tidak perlu.
Itu bagian dari keputusan yang diambil model, dan ia akan menjadi dasar
intent detection di 12.4.

## 5. Melihat mekanismenya secara langsung

Kita akan lihat apa yang sebenarnya terjadi di belakang layar saat kode melakukan tool calling

In [10]:
manual = [kalkulator]

resp = client.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Berapa 1247 dikali 89?"
        }
    ],
    tools=manual
)

# Membaca function call dari model
if resp.message.tool_calls:
    for panggilan in resp.message.tool_calls:
        print("nama fungsi :", panggilan.function.name)
        print("argumen     :", dict(panggilan.function.arguments))

        # TODO 4: Jalankan fungsi kalkulator secara manual
        if panggilan.function.name == "kalkulator":
            hasil = kalkulator(**panggilan.function.arguments)

            print("hasil lokal :", hasil)
else:
    print("Model tidak meminta pemanggilan tool.")
    print("Respons:", resp.message.content)

nama fungsi : kalkulator
argumen     : {'a': 1247, 'b': 89, 'operator': '*'}
hasil lokal : 110983



Tiga langkah, selalu berurutan, dan tidak pernah berubah:

1. Model **meminta**, dengan menyebut nama fungsi dan argumennya.
2. **Kode Anda** menjalankan fungsi itu.
3. Model **membaca** hasilnya dan menyusun jawaban akhir.

Model tidak pernah menjalankan kode apa pun. Ia hanya meminta. Kalau Anda tidak
menjalankan permintaannya, tidak ada yang terjadi.


In [11]:
# Kirim hasil kalkulator kembali ke model

messages = [
    {
        "role": "user",
        "content": "Berapa 1247 dikali 89?"
    },
    resp.message,
    {
        "role": "tool",
        "name": "kalkulator",
        "content": str(hasil)
    }
]

akhir = client.chat(
    model=MODEL,
    messages=messages,
    tools=[kalkulator]
)

print("Jawaban akhir:", akhir.message.content.strip())

Jawaban akhir: 1247 × 89 = 110 983




## 6. Di LangChain, hal yang sama dengan antarmuka yang berbeda

Kita memakai SDK Gemini secara langsung karena itu yang sudah Anda kenal sejak
Sesi 11, dan karena kodenya paling pendek.

Di kebanyakan codebase, Anda akan menemukan bentuk LangChain. Konsepnya persis
sama, hanya namanya berbeda:

```python
from langchain_core.tools import tool

@tool
def kalkulator(a: float, b: float, operator: str) -> str:
    """Hitung operasi aritmatika dua angka."""
    ...

llm_dengan_tool = llm.bind_tools([kalkulator])
pesan = llm_dengan_tool.invoke("Berapa 1247 dikali 89?")
print(pesan.tool_calls)
```

Keuntungannya, kode yang sama bisa dipindah ke model lain tanpa banyak berubah.
Biayanya, ada empat istilah baru sebelum baris pertama berjalan.

Sel di atas hanya untuk dibaca, tidak untuk dijalankan hari ini.

## 7. Giliran Anda

Tulis satu tool ketiga dari pekerjaan Anda sendiri, lalu tambahkan ke daftar dan
uji dengan pertanyaan yang seharusnya memanggilnya.

Ide yang mudah dikerjakan sekarang: menghitung usia dari tahun lahir, mengubah
mata uang dengan kurs tetap, atau menghitung cicilan dari pokok dan bunga.

In [14]:
# TODO 5: tulis satu tool ketiga milik Anda sendiri, lengkap dengan docstring
# dan type hint. Tambahkan ke daftar tools, lalu uji dengan pertanyaan yang
# seharusnya memanggilnya.

def kalkulator(a: float, b: float, operator: str) -> str:
    """Hitung operasi aritmatika dua angka.

    Args:
        a: Angka pertama.
        b: Angka kedua.
        operator: Salah satu dari '+', '-', '*', '/'.
    """
    if operator == "+":
        return str(a + b)
    if operator == "-":
        return str(a - b)
    if operator == "*":
        return str(a * b)
    if operator == "/":
        return str(a / b) if b != 0 else "Error: pembagian dengan nol"
    return "Error: operator tidak dikenal"


# TODO 5: tool ketiga
def hitung_cicilan(pokok: float, bunga_tahunan: float, tenor_bulan: int) -> str:
    """Hitung cicilan per bulan (bunga flat) dari pokok pinjaman.

    Args:
        pokok: Jumlah pinjaman dalam rupiah, misalnya 12000000.
        bunga_tahunan: Bunga per tahun dalam persen, misalnya 12 untuk 12%.
        tenor_bulan: Lama cicilan dalam bulan, misalnya 12.
    """
    total_bunga = pokok * (bunga_tahunan / 100) * (tenor_bulan / 12)
    cicilan = (pokok + total_bunga) / tenor_bulan
    return f"Cicilan per bulan: Rp{cicilan:,.0f}"

TOOLS = [kalkulator, hitung_cicilan]               # dikirim ke model
TOOL_MAP = {f.__name__: f for f in TOOLS}          # untuk mencari fungsi dari nama

def tanya(pertanyaan: str):
    messages = [{"role": "user", "content": pertanyaan}]

    resp = ollama.chat(model=MODEL, messages=messages, tools=TOOLS)
    messages.append(resp.message)

    # Selama model minta panggil tool, jalankan lalu kirim hasilnya
    while resp.message.tool_calls:
        for call in resp.message.tool_calls:
            nama = call.function.name
            args = call.function.arguments        # sudah berupa dict
            print(f"[tool] {nama}({args})")

            if nama in TOOL_MAP:
                hasil = TOOL_MAP[nama](**args)
            else:
                hasil = f"Error: tool {nama} tidak ada"

            messages.append(
                {"role": "tool", "tool_name": nama, "content": str(hasil)}
            )

        resp = ollama.chat(model=MODEL, messages=messages, tools=TOOLS)
        messages.append(resp.message)

    return resp

resp = tanya("Kalau pinjam 12 juta, bunga 12% per tahun, tenor 12 bulan, cicilannya berapa per bulan?")
print(resp.message.content.strip())

[tool] hitung_cicilan({'bunga_tahunan': 12, 'pokok': 12000000, 'tenor_bulan': 12})
Cicilan per bulan (bunga flat) untuk pinjaman Rp12 juta dengan suku bunga 12% per tahun dan tenor 12 bulan adalah **Rp1.120.000** per bulan.
